# exp091_self_gr_likelihood_pf_beam_probe train

Train-side candidate coverage and ranking audit for same-horizontal-well self-GR similarity candidates beside existing PF/Beam/likelihood-PF candidates.

## Contents

1. Setup and configuration
2. Exp072 feature cache and candidate schema
3. Self-GR candidate rank audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from self_gr_likelihood_pf_beam_probe import (
    FULL_REPLAY_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    run_from_config,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Cache parent:", cfg_get(config, "lineage.cache_parent"))
print("Diagnostic parents:", cfg_get(config, "lineage.diagnostic_parents"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Candidate specs:", [item["name"] for item in cfg_get(config, "audit.candidates", [])])
print("Self-GR candidate config:", cfg_get(config, "model.self_gr_candidate"))


## 2. Exp072 feature cache and candidate schema

In [ ]:
cache_path = find_artifact(
    FULL_REPLAY_TRAIN_FEATURES,
    cfg_get(config, "data.exp072_train_feature_cache_local"),
)
print("exp072 full replay train cache:", cache_path)
preview = pd.read_csv(cache_path, nrows=5, dtype={"id": str, "well": str})
candidate_source_cols = [item.get("source_column", item["name"]) for item in cfg_get(config, "audit.candidates", [])]
preview_cols = [
    c for c in ["id", "well", "target", "last_known_tvt", "md_since", "eval_len", *candidate_source_cols]
    if c in preview.columns
]
print("Columns:", len(preview.columns))
display(preview[preview_cols])


## 3. Self-GR candidate rank audit

In [ ]:
summary = run_from_config(config)
print(json.dumps(summary, indent=2)[:5000])


## 4. Metrics and artifacts

In [ ]:
candidate_metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_candidate_metrics.csv")
rank_metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_rank_metrics.csv")
bucket_metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_bucket_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_by_well.csv")
self_gr_well_summary = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_self_gr_well_summary.csv")

display(candidate_metrics)
display(rank_metrics)
display(bucket_metrics.head(60))
display(by_well.head(40))
display(self_gr_well_summary.describe(include="all"))
print("Candidate long:", paths.artifacts_dir / f"{OUTPUT_PREFIX}_candidate_long.csv.gz")
print("Summary:", paths.artifacts_dir / f"{OUTPUT_PREFIX}_summary.json")
